##### KnowledgeOS - pipeline walkthroughRuns the two implemented stages end to end and shows what each produces.```RAW SEC FILES  ->  INGESTION  ->  PostgreSQL records                                        |                                        v                                   PROCESSING  ->  CANONICAL DOCUMENTS```Not implemented yet, and deliberately absent below: chunking, embeddings,Qdrant indexing, retrieval, RAG, agents.**Prerequisites:** `docker compose up -d` (PostgreSQL + Qdrant healthy).This notebook runs on the host and reaches PostgreSQL on `localhost:5432`.

#### 0. Setup

In [1]:
import sys, os, json, logging
from pathlib import Path

# Repo root on the import path so `knowledgeos` and `services` resolve.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("repo root:", ROOT)

from knowledgeos.logging_setup import configure_logging

# Route logs to stdout so they render inline instead of as red stderr blocks.
configure_logging()
root_logger = logging.getLogger()
root_logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-7s %(name)-22s %(message)s"))
root_logger.addHandler(handler)
root_logger.setLevel(logging.INFO)

from knowledgeos.db.connection import check_connection
print(check_connection().split(",")[0])

repo root: C:\sasi\NeuroForge\KnowledgeOS


PostgreSQL 16.15 on x86_64-pc-linux-musl


In [2]:
import psycopg
from knowledgeos.db.connection import get_connection

TABLES = ["companies", "filings", "documents", "document_sections",
          "document_chunks", "vector_index_records", "processing_jobs"]

def counts():
    """Row counts for every table, as a quick state snapshot."""
    with get_connection(autocommit=True) as conn:
        return {t: conn.execute(f"SELECT count(*) AS n FROM {t}").fetchone()["n"]
                for t in TABLES}

def show(title="state"):
    c = counts()
    width = max(len(k) for k in c)
    print(title)
    for k, v in c.items():
        print(f"  {k:<{width}}  {v:>7,}")
    return c

show("current state")

current state
  companies                   0
  filings                     0
  documents                   0
  document_sections           0
  document_chunks             0
  vector_index_records        0
  processing_jobs             0


{'companies': 0,
 'filings': 0,
 'documents': 0,
 'document_sections': 0,
 'document_chunks': 0,
 'vector_index_records': 0,
 'processing_jobs': 0}

### Optional: start from scratchDrops and recreates the schema, then re-applies all migrations. **Destroys alldatabase rows.** Raw files under `data/raw/` are never touched.Uncomment to run.

In [3]:
# from knowledgeos.db.migrate import reset
# reset()
# show("after reset")

#### 1. IngestionReads the SEC metadata JSON already on disk, then for each filing:find-or-create the **company**, the **filing**, and the **document**, set thedocument to `DOWNLOADED`, and queue a `DOCUMENT_PROCESSING` job.`ingest_from_disk()` makes no network calls. (`ingest_from_sec()` is the samepipeline with a live EDGAR fetch in front.)

In [4]:
from services.ingestion.pipeline import ingest_from_disk

# Quieten the per-filing chatter; the summary line is what matters here.
logging.getLogger("ingestion.pipeline").setLevel(logging.WARNING)

stats = ingest_from_disk()
print()
print("summary:", stats.summary())
show("after ingestion")


summary: companies +10/=35  filings +45/=0  documents +45/=0  failed=0  jobs +45/=0


after ingestion
  companies                  10
  filings                    45
  documents                  45
  document_sections           0
  document_chunks             0
  vector_index_records        0
  processing_jobs            45


{'companies': 10,
 'filings': 45,
 'documents': 45,
 'document_sections': 0,
 'document_chunks': 0,
 'vector_index_records': 0,
 'processing_jobs': 45}

Note the counters: `+N` means created, `=N` means found. Run this cell a secondtime and everything becomes `=` - ingestion is idempotent, keyed on CIK andaccession number.

In [5]:
with get_connection(autocommit=True) as conn:
    rows = conn.execute("""
        SELECT c.ticker, c.name, count(DISTINCT f.id) AS filings, count(d.id) AS docs
          FROM companies c
          LEFT JOIN filings f ON f.company_id = c.id
          LEFT JOIN documents d ON d.filing_id = f.id
         GROUP BY c.id, c.ticker, c.name ORDER BY c.name
    """).fetchall()

print(f"{'TICKER':<8}{'FILINGS':>8}{'DOCS':>6}  NAME")
for r in rows:
    print(f"{(r['ticker'] or '-'):<8}{r['filings']:>8}{r['docs']:>6}  {r['name']}")

TICKER   FILINGS  DOCS  NAME
AMD            5     5  ADVANCED MICRO DEVICES INC
AMZN           5     5  AMAZON COM INC
GOOGL          3     3  Alphabet Inc.
AAPL           5     5  Apple Inc.
INTC           5     5  INTEL CORP
MSFT           5     5  MICROSOFT CORP
META           2     2  Meta Platforms, Inc.
NVDA           5     5  NVIDIA CORP
ORCL           5     5  ORACLE CORP
TSLA           5     5  Tesla, Inc.


#### 2. ProcessingFor each document: detect the format, resolve a parser from the registry,parse into a **canonical document**, write it to `data/processed/`, persist thestructural outline, and move both the document and its job through theirstates.```document:  DOWNLOADED -> PROCESSING -> PROCESSED | FAILEDjob:       QUEUED      -> PROCESSING -> COMPLETED | FAILED```Starting with three documents so the log is readable.

In [6]:
from services.processing.pipeline import process_pending

logging.getLogger("processing.pipeline").setLevel(logging.INFO)

pstats = process_pending(limit=3)
print()
print("summary:", pstats.summary())

INFO    processing.pipeline    Processing stage starting | parsers=html v2.0.0


INFO    processing.pipeline    3 document(s) to process


INFO    processing.pipeline    Job PROCESSING: 2024-01-31_0001652044-24-000022_goog-20231231.htm


INFO    processing.pipeline    Document PROCESSING: 2024-01-31_0001652044-24-000022_goog-20231231.htm


INFO    processing.pipeline    Detected format: HTML (2024-01-31_0001652044-24-000022_goog-20231231.htm)


...

INFO    processing.pipeline    Parser: html v2.0.0


INFO    processing.pipeline    Document PROCESSED: 2026-02-05_0001652044-26-000018_goog-20251231.htm -> data/processed/sec/Alphabet/10-K/2026-02-05_0001652044-26-000018_goog-20251231.htm.canonical.json | nodes=1332 sections=284 paragraphs=867 tables=181 lists=0 code=0 quotes=0 chars=339373 depth=3


INFO    processing.pipeline    Job COMPLETED: 2026-02-05_0001652044-26-000018_goog-20251231.htm


INFO    processing.pipeline    Processing stage complete | processed=3 skipped=0 failed=0 sections=872



summary: processed=3 skipped=0 failed=0 sections=872


Now the rest.

In [7]:
logging.getLogger("processing.pipeline").setLevel(logging.WARNING)
pstats = process_pending()
print("summary:", pstats.summary())
show("after processing")

summary: processed=42 skipped=0 failed=0 sections=9415


after processing
  companies                  10
  filings                    45
  documents                  45
  document_sections      10,287
  document_chunks             0
  vector_index_records        0
  processing_jobs            45


{'companies': 10,
 'filings': 45,
 'documents': 45,
 'document_sections': 10287,
 'document_chunks': 0,
 'vector_index_records': 0,
 'processing_jobs': 45}

#### 3. The canonical documentThe format-independent output. A tree of generic **nodes** - `section` is onenode type among several, not something every document must have.

In [8]:
from services.processing.storage import read_canonical

with get_connection(autocommit=True) as conn:
    row = conn.execute("""
        SELECT file_name, storage_path, processed_path, processor_name, processor_version
          FROM documents WHERE status = 'PROCESSED' AND file_name LIKE '%aapl%'
         ORDER BY file_name DESC LIMIT 1
    """).fetchone()

doc = read_canonical(row["processed_path"])

print("raw       ", row["storage_path"])
print("canonical ", row["processed_path"])
print("processor ", f"{row['processor_name']} v{row['processor_version']}")
print("schema    ", doc.schema_version)
print()
print("metadata carried from ingestion:")
for k in ("company_name", "cik", "ticker", "form_type", "filing_date", "report_date"):
    if doc.metadata.get(k):
        print(f"  {k:<14} {doc.metadata[k]}")
print()
print("stats:", {k: v for k, v in doc.stats().items() if v})

raw        data/raw/sec/Apple/10-K/2025-10-31_0000320193-25-000079_aapl-20250927.htm
canonical  data/processed/sec/Apple/10-K/2025-10-31_0000320193-25-000079_aapl-20250927.htm.canonical.json
processor  html v2.0.0
schema     2.0

metadata carried from ingestion:
  company_name   Apple Inc.
  cik            0000320193
  ticker         AAPL
  form_type      10-K
  filing_date    2025-10-31
  report_date    2025-09-27

stats: {'nodes': 698, 'section': 170, 'paragraph': 474, 'table': 54, 'text_length': 203663, 'max_depth': 3}


In [9]:
from services.processing.canonical import NodeType

print("has sections :", doc.has_sections())
print("root nodes   :", len(doc.content))
print()
print("outline (first 20 nodes):")
for i, (node, parent) in enumerate(doc.iter_with_parents()):
    if i >= 20:
        break
    indent = "  " * ((node.level or 1) - 1 if node.type is NodeType.SECTION else 1)
    extra = ""
    if node.type is NodeType.TABLE and node.table:
        extra = f" [{node.table.n_rows}x{node.table.n_cols}]"
    elif node.type is NodeType.SECTION:
        extra = f" [L{node.level}, {len(node.children)} children]"
    print(f"  {indent}<{node.type.value}>{extra} {node.text[:56]}")

has sections : True
root nodes   : 4

outline (first 20 nodes):
      <section> [L3, 0 children] UNITED STATES
      <section> [L3, 1 children] SECURITIES AND EXCHANGE COMMISSION
        <section> [L4, 0 children] Washington, D.C. 20549
    <section> [L2, 7 children] FORM 10-K
    <paragraph> (Mark One)
    <paragraph> ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE S
    <paragraph> For the fiscal year ended September 27 , 2025
    <paragraph> or
    <paragraph> ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF T
    <paragraph> For the transition period from to .
    <paragraph> Commission File Number: 001-36743
  <section> [L1, 167 children] Apple Inc.
    <paragraph> (Exact name of Registrant as specified in its charter)
    <table> [5x2] 
          <section> [L5, 22 children] ( 408 ) 996-1010
    <paragraph> (Registrant’s telephone number, including area code)
    <paragraph> Securities registered pursuant to Section 12(b) of the A
    <table> [9x3] 
    <paragraph> Se

### Structure is preserved, not flattenedA table stays rows and columns. This is the point of the canonical layer:chunking later can decide what to do with a table, because the table is stilla table.

In [10]:
tables = doc.tables()
print(f"{len(tables)} tables in this document")

# Pick the widest table with real content.
t = max(tables, key=lambda x: (x.n_cols, x.n_rows))
print(f"\nshowing a {t.n_rows}x{t.n_cols} table")
if t.caption:
    print("caption:", t.caption)
if t.header:
    print("  H | " + " | ".join(c[:20] for c in t.header))
for r in t.rows[:8]:
    print("    | " + " | ".join(c[:20] for c in r))

54 tables in this document

showing a 18x15 table
    |  | 2025 |  |  |  |  |  |  |  |  |  |  |  |  | 
    |  | Adjusted Cost |  | Unrealized Gains |  | Unrealized Losses |  | Fair Value |  | Cash and Cash Equiva |  | Current Marketable S |  | Non-Current Marketab | 
    | Cash | $ | 28,267 | $ | — | $ | — | $ | 28,267 | $ | 28,267 | $ | — | $ | —
    | Level 1: |  |  |  |  |  |  |  |  |  |  |  |  |  | 
    | Money market funds | 5,272 |  | — |  | — |  | 5,272 |  | 5,272 |  | — |  | — | 
    | Mutual funds | 679 |  | 177 |  | ( 2 ) |  | 854 |  | — |  | 854 |  | — | 
    | Subtotal | 5,951 |  | 177 |  | ( 2 ) |  | 6,126 |  | 5,272 |  | 854 |  | — | 
    | Level 2 (1) : |  |  |  |  |  |  |  |  |  |  |  |  |  | 


In [11]:
# Raw JSON for one section node - what is actually on disk.
section = next(n for n in doc.walk()
               if n.type is NodeType.SECTION and len(n.children) >= 2)

def trim(n, depth=0):
    o = {"id": n.id, "type": n.type.value}
    if n.text:
        o["text"] = n.text[:60] + ("..." if len(n.text) > 60 else "")
    if n.attributes:
        o["attributes"] = n.attributes
    if n.table:
        o["table"] = {"n_rows": n.table.n_rows, "n_cols": n.table.n_cols,
                      "rows": [r[:3] for r in n.table.rows[:2]]}
    if n.children and depth < 1:
        o["children"] = [trim(c, depth + 1) for c in n.children[:3]]
    return o

print(json.dumps(trim(section), indent=2, ensure_ascii=False))

{
  "id": "n-0004",
  "type": "section",
  "text": "FORM 10-K",
  "attributes": {
    "level": 2,
    "path": []
  },
  "children": [
    {
      "id": "n-0005",
      "type": "paragraph",
      "text": "(Mark One)"
    },
    {
      "id": "n-0006",
      "type": "paragraph",
      "text": "☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECUR..."
    },
    {
      "id": "n-0007",
      "type": "paragraph",
      "text": "For the fiscal year ended September 27 , 2025"
    }
  ]
}


### Plain text is a derived view`text()` renders the tree. It is never the stored form - the structure above is.

In [12]:
print(doc.text()[:600], "...")

UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM 10-K

(Mark One)

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 27 , 2025

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from to .

Commission File Number: 001-36743

Apple Inc.

(Exact name of Registrant as specified in its charter)

California | 94-2404110

(State or other jurisdiction of incorporation or organization) | (I.R.S. Employer Identification No.)

One ...


#### 4. IdempotencyRe-running processing does nothing: every document already matches the currentprocessor name and version.

In [13]:
logging.getLogger("processing.pipeline").setLevel(logging.INFO)
process_pending()

INFO    processing.pipeline    Processing stage starting | parsers=html v2.0.0


INFO    processing.pipeline    Nothing to process; all documents are current


ProcessingStats(processed=0, skipped=0, failed=0, sections_written=0, errors=[])

#### 5. Version-triggered reprocessingProcessor identity is stored per document. Pretending one document wasprocessed by an older version makes exactly that document eligible again -nothing else moves.

In [14]:
with get_connection() as conn:
    changed = conn.execute("""
        UPDATE documents SET processor_version = '0.0.1'
         WHERE file_name LIKE '%nvda%' RETURNING file_name
    """).fetchall()
print(f"marked {len(changed)} document(s) as processed by an old version\n")

process_pending()

marked 5 document(s) as processed by an old version

INFO    processing.pipeline    Processing stage starting | parsers=html v2.0.0


INFO    processing.pipeline    5 document(s) to process


WARNING processing.pipeline    No open DOCUMENT_PROCESSING job for 2022-03-18_0001045810-22-000036_nvda-20220130.htm; processing anyway


INFO    processing.pipeline    Document PROCESSING: 2022-03-18_0001045810-22-000036_nvda-20220130.htm


INFO    processing.pipeline    Detected format: HTML (2022-03-18_0001045810-22-000036_nvda-20220130.htm)


...

INFO    processing.pipeline    Detected format: HTML (2026-02-25_0001045810-26-000021_nvda-20260125.htm)


INFO    processing.pipeline    Parser: html v2.0.0


INFO    processing.pipeline    Document PROCESSED: 2026-02-25_0001045810-26-000021_nvda-20260125.htm -> data/processed/sec/NVIDIA/10-K/2026-02-25_0001045810-26-000021_nvda-20260125.htm.canonical.json | nodes=1200 sections=294 paragraphs=842 tables=64 lists=0 code=0 quotes=0 chars=336381 depth=4


INFO    processing.pipeline    Processing stage complete | processed=5 skipped=0 failed=0 sections=1665


ProcessingStats(processed=5, skipped=0, failed=0, sections_written=1665, errors=[])

#### 6. Format detection and the parser registryDetection is content-first: file contents are trusted ahead of the extension,because extensions lie. The registry is how a new format gets added withouttouching the pipeline.

In [15]:
from services.processing.detection import detect_format, sniff
from services.processing.base import registry

samples = {
    "filing.htm":  b"<html><body><p>hi</p></body></html>",
    "report.txt":  b"<html><body>HTML served as .txt</body></html>",
    "notes.txt":   b"just prose, no markup",
    "paper.pdf":   b"%PDF-1.7\n...",
    "data.json":   b'{"a": 1}',
    "readme.md":   b"# Title\n<div>inline html</div>",
    "mystery.zzz": b"\x00\x01binary",
}
for name, head in samples.items():
    print(f"  {name:<14} -> {detect_format(name, head=head).value}")

print()
print("registered parsers:")
for p in registry.parsers():
    fmts = ", ".join(sorted(f.value for f in p.supported_formats))
    print(f"  {p.name} v{p.version}  handles: {fmts}")
print()
print("formats with no parser yet:", "PDF, DOCX, TXT, MARKDOWN, JSON, CSV, XML")

  filing.htm     -> HTML
  report.txt     -> HTML
  notes.txt      -> TXT
  paper.pdf      -> PDF
  data.json      -> JSON
  readme.md      -> MARKDOWN
  mystery.zzz    -> UNKNOWN

registered parsers:
  html v2.0.0  handles: HTML

formats with no parser yet: PDF, DOCX, TXT, MARKDOWN, JSON, CSV, XML


#### 7. The parser is genericThe same HTML parser, given a document that is nothing like a filing. No SECknowledge is involved - it reacts to HTML structure only.

In [16]:
from services.processing.parsers.html_parser import HtmlParser
from services.processing.canonical import SourceInfo

parser = HtmlParser()
src = SourceInfo(path="demo", file_name="demo.html", doc_format="HTML")

article = b"""<html><body>
  <h1>On Retrieval</h1>
  <h2>Abstract</h2><p>We study retrieval quality.</p>
  <h2>Method</h2>
  <p>We ran the following:</p>
  <pre><code class="language-python">index.search(q, k=10)</code></pre>
  <ul><li>dense</li><li>sparse</li></ul>
  <blockquote>Prior work disagrees.</blockquote>
</body></html>"""

a = parser.parse(src, article)
print("has sections:", a.has_sections())
for n in a.walk():
    print(f"  <{n.type.value}> {n.text[:50]}")

has sections: True
  <section> On Retrieval
  <section> Abstract
  <paragraph> We study retrieval quality.
  <section> Method
  <paragraph> We ran the following:
  <code> index.search(q, k=10)
  <list> 
  <list_item> dense
  <list_item> sparse
  <quote> Prior work disagrees.


In [17]:
# A flat document - no headings at all - stays flat. No synthetic section
# is invented just to satisfy a shape.
flat = b"""<html><body>
  <p>Install the package.</p>
  <pre>pip install knowledgeos</pre>
  <table><tr><th>Flag</th></tr><tr><td>--fast</td></tr></table>
</body></html>"""

f = parser.parse(src, flat)
print("has sections:", f.has_sections())
print("root nodes  :", [n.type.value for n in f.content])

has sections: False
root nodes  : ['paragraph', 'code', 'table']


#### 8. Raw files were never modifiedProcessing reads raw files and writes elsewhere. Re-hashing every raw file andcomparing against the checksum recorded at ingestion proves it.

In [18]:
import hashlib
from knowledgeos.config import REPO_ROOT

with get_connection(autocommit=True) as conn:
    rows = conn.execute(
        "SELECT file_name, storage_path, checksum_sha256 FROM documents "
        "WHERE checksum_sha256 IS NOT NULL"
    ).fetchall()

bad = [r["file_name"] for r in rows
       if hashlib.sha256((REPO_ROOT / r["storage_path"]).read_bytes()).hexdigest()
       != r["checksum_sha256"]]

print(f"re-hashed {len(rows)} raw files")
print(f"mismatches: {len(bad)}")
print("data/raw is", "UNCHANGED" if not bad else f"MODIFIED: {bad}")

re-hashed 45 raw files
mismatches: 0
data/raw is UNCHANGED


#### Where this stops`document_chunks` and `vector_index_records` are still empty, and Qdrant has nocollections. That is the next stage, not this one.

In [19]:
c = show("final state")
print()
print("still empty, by design:",
      [k for k in ("document_chunks", "vector_index_records") if c[k] == 0])

final state
  companies                  10
  filings                    45
  documents                  45
  document_sections      10,287
  document_chunks             0
  vector_index_records        0
  processing_jobs            45

still empty, by design: ['document_chunks', 'vector_index_records']
